Transformation des fichiers stéréo en mono

In [ ]:
import numpy as np
import os
import librosa
import soundfile as sf
from glob import glob
import librosa as lr

from os import listdir
from os.path import isfile, join
from pathlib import Path
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt

import IPython



In [ ]:
#If you use a drive
#from google.colab import drive
#drive.mount('/content/drive')

#Initial sound loading
#data_dir_ini = '/content/drive/MyDrive/Article sound/audio/0_raw_sound/'

Mounted at /content/drive


In [ ]:
#If you use work on github
from pathlib import Path
from glob import glob

# Repository root directory
repo_dir = Path(__file__).resolve().parent

# Directory containing the raw sound files
data_dir_ini = repo_dir / "audio" / "0_raw_sound"

# List of all raw .wav files
audio_files_ini = sorted(glob(str(data_dir_ini / "*.wav")))

First step: converting the sounds in the repertoire to mono

In [ ]:
#Converting sounds from the repertoire to mono
sounds_name_ini = [f for f in listdir(data_dir_ini) if isfile(join(data_dir_ini, f))and f.endswith(".wav")]

for sound in sounds_name_ini:
    path = data_dir_ini+sound

    y, sr = librosa.load(path, sr=None, mono=False)
    y_mono = y[0, :]
    file_name_out = "/content/drive/MyDrive/Article sound/audio/1_mono/" + Path(sound).stem.replace('_ini','') + "_mono.wav" # to remove "_ini"
    sf.write(file_name_out, y_mono, sr)
print("Sounds converted to mono -> /audio/1_mono/")

Sounds converted to mono -> /audio/1_mono/


Second step: noise filtering

In [ ]:
#Noise filtering function
def filter_sound (audio, cutoff, order):

    sounds_name = [f for f in listdir(audio) if isfile(join(audio, f)) and f.endswith(".wav")]

    for sound in sounds_name:

        sound_name = audio + sound

        y, sr = librosa.load(sound_name, sr=None)

        #Sound is previously normalized
        max_val = np.max(np.abs(y))
        y_norm = y / max_val

        #Passe haut
        nyquist = 0.5 * sr
        normal_cutoff = cutoff / nyquist
        b, a = butter(order, normal_cutoff, btype='high', analog=False)
        filtered_audio = filtfilt(b, a, y_norm)

        #Extraction du fichier
        file_name_out = "/content/drive/MyDrive/Article sound/audio/2_filtered/" + sound.replace("_mono", "_filtered")

        sf.write(file_name_out, filtered_audio, sr )


In [ ]:
#Noise filtering
cutoff = 100  # Cut-off frequency in Hz

order = 4  # Filter order

data_dir_ini = '/content/drive/MyDrive/Article sound/audio/1_mono/'

filter_sound (data_dir_ini, cutoff, order)
print("Sounds filtered -> ./audio/2_filtered/")

Sounds filtered -> ./audio/2_filtered/


Third step: time normalization

In [ ]:
#Function for time normalization
def remove_silence(data, threshold, duration, lag, attack):

    sounds_name = [f for f in listdir(data) if isfile(join(data, f)) and f.endswith(".wav")]

    for sound in sounds_name:
        y, sr = librosa.load(data+sound, sr=None)

        #Sound is previously normalized
        max_val = np.max(np.abs(y))
        y_norm = y / max_val

        #Sound is then extruded
        raw_start = np.min(np.where(np.abs(y_norm)>threshold))
        min_near_first = np.argmin(np.abs(y_norm[raw_start-lag : raw_start]))
        start = raw_start - lag + min_near_first
        sig_without_silence = y_norm[start : start + duration]
        sig_without_attack = sig_without_silence[int(attack*sr) : ]

        file_name_out = "/content/drive/MyDrive/Article sound/audio/3_trim/" + sound.replace("_filtered", "_trimmed")

        sf.write(file_name_out, sig_without_attack, sr)

In [ ]:
#Time normalization after filtering
data_dir_ini = '/content/drive/MyDrive/Article sound/audio/2_filtered/' #loading filtered files

threshold = 0.7e-01

duration = int(0.25 * sr)  # duration 250 ms
lag = 5 # in samples

attack = 0.009 #definition of the attack duration

remove_silence(data_dir_ini, threshold, duration, lag, attack) #Silence removal
print("Sounds trimmed -> ./audio/3_trim/")

Sounds trimmed -> ./audio/3_trim/


Quatrième étape: normalisation en amplitude

In [ ]:
#Amplitude normalization function
def ampl_normalize_audio(data_dir):

    sounds_name = [f for f in listdir(data_dir) if isfile(join(data_dir, f)) and f.endswith(".wav")]

    for sound in sounds_name:
        sound_name = data_dir + sound

        y, sr = librosa.load(sound_name, sr=None)

        max_val = np.max(np.abs(y))
        normalized_audio = y / max_val

        file_name_out = "/content/drive/MyDrive/Article sound/audio/4_norm/" + sound.replace("_trimmed", "_norm")

        sf.write(file_name_out, normalized_audio, sr )

In [ ]:
#Amplitude normalization
data_dir_ini = '/content/drive/MyDrive/Article sound/audio/3_trim/'

ampl_normalize_audio(data_dir_ini)
print("Sounds normalized -> ./audio/4_norm/")

Sounds normalized -> ./audio/4_norm/
